# Part 3: Autograd and the computational graph

Part 1 derived the gradients by hand. PyTorch records each operation on tensors that have `requires_grad=True`, and replays the record backwards to get every gradient.

In [1]:
import torch

x = torch.tensor(2.0)                         # an input
t = torch.tensor(10.0)                        # the target
w = torch.tensor(3.0, requires_grad=True)     # a parameter to learn
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b                                 # forward
loss = (y - t) ** 2
print("prediction:", y.item(), " loss:", loss.item())
print("how the loss was made:", loss.grad_fn)
loss.backward()                               # backward
print("d loss / d w =", w.grad.item(), "  (by hand: 2 * (y - t) * x = 2 * (7 - 10) * 2 = -12)")
print("d loss / d b =", b.grad.item(), "  (by hand: 2 * (y - t) = -6)")

prediction: 7.0  loss: 9.0
how the loss was made: <PowBackward0 object at 0x7f62a2a79120>
d loss / d w = -12.0   (by hand: 2 * (y - t) * x = 2 * (7 - 10) * 2 = -12)
d loss / d b = -6.0   (by hand: 2 * (y - t) = -6)


## Gradients accumulate

Each call to `backward()` adds to `.grad`. That is why a training loop must clear the gradients before every step.

In [2]:
w = torch.tensor(3.0, requires_grad=True)
for call in (1, 2, 3):
    loss = (w * 2.0 + 1.0 - 10.0) ** 2
    loss.backward()
    print(f"after backward call {call}: w.grad = {w.grad.item()}")
w.grad.zero_()
print("after zero_():", w.grad.item())

after backward call 1: w.grad = -12.0
after backward call 2: w.grad = -24.0
after backward call 3: w.grad = -36.0
after zero_(): 0.0


## Switching the record off

When you only want predictions, tracking is wasted work. `torch.no_grad()` turns it off, and `.detach()` cuts one tensor out of the graph.

In [3]:
w = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
    y = w * 2.0
print("inside no_grad, requires_grad:", y.requires_grad)
z = (w * 2.0).detach()
print("detached, requires_grad:", z.requires_grad)
try:
    (w * 2.0).numpy()
except RuntimeError as err:
    print("error:", err)

inside no_grad, requires_grad: False
detached, requires_grad: False
error: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.


## The Part 1 line fit again, with autograd

The two hand-derived gradient lines are replaced by one call. The loop, the numbers and the result are the same as in Part 1.

In [4]:
import numpy as np

rng = np.random.default_rng(0)
x = torch.tensor(rng.uniform(0, 5, 50), dtype=torch.float32)
y = 3 * x + 2 + torch.tensor(rng.normal(0, 0.3, 50), dtype=torch.float32)

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.05

for step in range(1, 201):
    loss = ((w * x + b - y) ** 2).mean()
    loss.backward()                         # replaces the hand-derived gradients
    with torch.no_grad():                   # the update itself must not be recorded
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()
    if step in (1, 2, 3, 10, 50, 100, 200):
        print(f"step {step:3d}  loss {loss.item():8.4f}  w {w.item():.3f}  b {b.item():.3f}")

step   1  loss 117.8619  w 3.255  b 0.991
step   2  loss   0.3161  w 3.299  b 1.026
step   3  loss   0.2800  w 3.294  b 1.046
step  10  loss   0.2299  w 3.257  b 1.170
step  50  loss   0.1140  w 3.126  b 1.611
step 100  loss   0.0922  w 3.063  b 1.821
step 200  loss   0.0894  w 3.035  b 1.915


## Your turn

1. Comment out the two `zero_()` lines and rerun. Explain what you see.
2. Remove the `with torch.no_grad():` line (keep the indented lines). What error do you get, and why?
3. Compute the gradient of `w ** 3` at `w = 2` with autograd, and check it against `3 * w ** 2`.